# GIAI ĐOẠN 3: DATA PREPARATION, EXPLORATORY DATA ANALYSIS & DATA VISUALIZATION
**Đề tài:** Phân tích Biểu hiện Định giá Động (Dynamic Surge Pricing) và Dự báo Giá báo API trên Nền tảng Gọi xe Công nghệ (Uber & Lyft Boston)  
**Mã đề tài:** `PRJ-EXT-01` | **Sinh viên thực hiện:** Trần Thanh Huy (`2521001060`) & Lê Huy Thắng (`2521001092`)  
**Môi trường thực thi:** Python `3.11.9` | `pandas 3.1.1` | `numpy 2.4.6` | `scipy 1.17.1` | `statsmodels 0.15.0` | `scikit-learn 1.9.1`

---
## PHẦN 1: DATA PREPARATION – KIỂM TRA BAN ĐẦU & LÀM SẠCH DỮ LIỆU
### Bước 1.1. Đọc dữ liệu gốc và Kiểm tra trạng thái ban đầu (Initial Data Audit)
- **Mục đích:** Đọc nguyên trạng 2 bảng `cab_rides.csv` và `weather.csv` (chế độ chỉ đọc) để ghi nhận số dòng/cột, kiểu dữ liệu, khóa chính/ngoại, tỷ lệ khuyết thiếu, trùng lặp và 7 tầng bẫy nhiễu.
- **Lý do chọn phương pháp:** Quét bề mặt kết hợp biểu thức chính quy (Regex) trên bản sao tạm giúp phát hiện cả các chuỗi báo khuyết ẩn (`'?'`, `'-'`, `'chưa rõ'`) và mã lính canh (`9999`, `-1`, `-100`) mà không làm thay đổi tệp gốc.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for extra_path in (BASE_DIR, BASE_DIR / "src"):
    if str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

cab_raw = pd.read_csv(RAW_DIR / "cab_rides.csv", low_memory=False)
weather_raw = pd.read_csv(RAW_DIR / "weather.csv", low_memory=False)

cab_mem = cab_raw.memory_usage(deep=True).sum() / 1024**2
wea_mem = weather_raw.memory_usage(deep=True).sum() / 1024**2
print(f"cab_rides.csv shape: {cab_raw.shape} | Memory: {cab_mem:.2f} MiB")
print(f"weather.csv shape  : {weather_raw.shape} | Memory: {wea_mem:.2f} MiB")
print("\ncab_rides raw nulls:\n", cab_raw.isna().sum())
print("\nweather raw nulls:\n", weather_raw.isna().sum())

cab_rides.csv shape: (693071, 10) | Memory: 126.21 MiB
weather.csv shape  : (6276, 8) | Memory: 0.64 MiB

cab_rides raw nulls:
 distance             28413
cab_type            129294
time_stamp          115023
destination          97701
source               79802
price               144638
surge_multiplier     99132
id                       0
product_id               0
name                108628
dtype: int64

weather raw nulls:
 temp           439
location       273
clouds         637
pressure      1104
rain          4307
time_stamp     597
humidity       403
wind           512
dtype: int64


### Bước 1.2 & 1.3. Thực thi Pipeline Làm sạch Dữ liệu (`cab_rides` & `weather`)
- **Mục đích:**
  1. Khôi phục tất định 100% `cab_type` (`466,242` dòng bẩn/khuyết) và `name` (`548,168` dòng bẩn/khuyết) từ khóa nội tại `product_id`.
  2. Chuẩn hóa 12 quận Boston (`source`, `destination`, `location`) bằng `.str.strip().str.title()` và sửa lỗi cắt ngắn `'Northeastern Universi'`.
  3. Bóc tách hậu tố đơn vị (`USD`, `VND`, `đ`, `kg`, `%`), đổi dấu phẩy thập phân `,` thành `.`, chuyển chuỗi khuyết ẩn và mã lính canh (`-1`, `-100`, `1000`, `9999`, `10000`, `100000`) về `np.nan`.
  4. Khôi phục dấu âm (`abs()`) và khôi phục tỷ lệ phóng đại bậc 10 ($10\times, 100\times, 1000\times$) theo công thức $\log_{10}$ (cả trên `cab_rides` lẫn khí tượng `weather` theo trung vị cùng giờ toàn Boston).
  5. Loại bỏ dòng mất biến mục tiêu `price` (`55,095` dòng `Uber Taxi` MNAR) hoặc mất lộ trình; **KHÔNG điền mù `surge_multiplier = 1.0` trên `38,037` dòng Lyft khuyết surge** (gắn cờ `is_surge_original`) và giữ nguyên `rain = NaN` trên `1,074` bản tin khí tượng lỗi cảm biến mưa (gắn cờ `is_rain_sentinel`).

In [2]:
from src.clean_raw_data import clean_cab_rides, clean_weather

cab_clean = clean_cab_rides(cab_raw)
weather_clean = clean_weather(weather_raw)

ts_n = int(cab_clean["is_timestamp_original"].sum())
ts_pct = cab_clean["is_timestamp_original"].mean() * 100
print(
    f"cab_rides sau làm sạch: {cab_clean.shape} | "
    f"TS gốc hợp lệ: {ts_n:,} ({ts_pct:.2f}%)"
)
w_nulls = int(weather_clean.isna().sum().sum())
print(
    f"weather sau làm sạch  : {weather_clean.shape} | Nulls còn lại: {w_nulls}"
)
print(cab_clean[["distance", "surge_multiplier", "price"]].describe().round(3))

2026-11-28 03:42:04,588 - INFO - === BẮT ĐẦU LÀM SẠCH CAB_RIDES.CSV (CHUNKED STREAMING) ===


2026-11-28 03:42:08,430 - INFO - Số dòng cab_rides hợp lệ sau khi lọc missing price/route: 270,898 / 693,071


2026-11-28 03:42:08,434 - INFO - [CRIT-03] Kiểm toán surge_multiplier Lyft: Quan sát thực = 92,384, Khuyết/Sentinel = 38,037 (29.16%).


2026-11-28 03:42:08,440 - INFO - [CRIT-02] Kiểm toán time_stamp: Gốc hợp lệ = 208,034 (76.79%), Thiếu (giữ NA) = 62,864.


2026-11-28 03:42:08,494 - INFO - === BẮT ĐẦU LÀM SẠCH WEATHER.CSV (H4: LOG10 + SAME-HOUR MEDIAN + RAIN SENTINEL) ===


2026-11-28 03:42:08,587 - INFO - Hoàn thành làm sạch weather: 5,053 dòng (rain sentinels giữ NaN: 1,074).


cab_rides sau làm sạch: (270898, 12) | TS gốc hợp lệ: 208,034 (76.79%)
weather sau làm sạch  : (5053, 9) | Nulls còn lại: 1074
         distance  surge_multiplier       price
count  248636.000        232861.000  270898.000
mean        2.197             1.012      16.576
std         1.141             0.086       9.346
min         0.020             1.000       2.500
25%         1.280             1.000       9.000
50%         2.170             1.000      13.500
75%         2.940             1.000      22.500
max         7.860             3.000      92.000


---
## PHẦN 2: EXPLORATORY DATA ANALYSIS (EDA) & GHÉP NỐI ĐA BẢNG
### Bước 2.1 – 2.4. Ghép nối Không gian – Thời gian (`Spatio-Temporal Left Join` theo Boston EST) và Thống kê Mô tả
- **Mục đích:** Chuyển đổi timestamp hợp lệ từ UTC sang múi giờ địa phương Boston (`America/New_York`, EST = UTC-5), KHÔNG suy đoán `ffill` cho `62,864` dòng thiếu timestamp gốc; gom nhóm `weather_clean` theo khóa `(location, date_hour)` (`5,053 -> 3,365` mốc duy nhất) trước khi `LEFT JOIN` vào `cab_clean`, đảm bảo **0 dòng bị nhân bản (`270,898` trước Join = `270,898` sau Join, `32` cột, RAM `37.46 MiB`)**, gắn cờ `weather_matched` (`179,280` dòng khớp trạm giờ), `is_rain_sentinel` (`27,237` dòng ô sentinel mưa; `152,043` dòng mưa/khô hợp lệ) và `is_surge_original` (`232,861` dòng có surge quan sát thực).

In [3]:
master_df = pd.read_parquet(PROCESSED_DIR / "master_rideshare.parquet")
m_mem = master_df.memory_usage(deep=True).sum() / 1024**2
print(f"Master table shape: {master_df.shape} | Memory: {m_mem:.2f} MiB")
ts_ok = int((master_df["is_timestamp_original"] == 1).sum())
wm_ok = int((master_df["weather_matched"] == 1).sum())
print(f"TS gốc: {ts_ok:,} | Khớp thời tiết thực: {wm_ok:,}")
num_cols = [
    "price",
    "distance",
    "surge_multiplier",
    "temp",
    "clouds",
    "pressure",
    "rain",
    "humidity",
    "wind",
]
stats_cols = ["count", "mean", "50%", "std", "min", "25%", "75%", "max"]
print(master_df[num_cols].describe().T[stats_cols].round(3))

Master table shape: (270898, 32) | Memory: 37.46 MiB
TS gốc: 208,034 | Khớp thời tiết thực: 179,280
                     count      mean       50%     std     min      25%  \
price             270898.0    16.576    13.500   9.346    2.50    9.000   
distance          248636.0     2.197     2.170   1.141    0.02    1.280   
surge_multiplier  232861.0     1.012     1.000   0.086    1.00    1.000   
temp              179280.0    39.212    40.210   6.517   19.62   36.285   
clouds            179280.0     0.673     0.770   0.313    0.00    0.430   
pressure          179280.0  1009.551  1008.133  13.495  988.25  998.675   
rain              152043.0     0.007     0.000   0.034    0.00    0.000   
humidity          179280.0     0.750     0.730   0.129    0.45    0.650   
wind              179280.0     7.035     6.860   3.642    0.30    3.730   

                       75%       max  
price               22.500    92.000  
distance             2.940     7.860  
surge_multiplier     1.000     3

---
## PHẦN 3 & PHẦN 4: TRỰC QUAN HÓA DỮ LIỆU & GIẢI QUYẾT 3 BÀI TOÁN KINH DOANH
### Kiểm định Thống kê (Bài toán 1), Phân rã Cấu trúc Giá OLS (Bài toán 2) & Mô hình Dự báo (Bài toán 3)


In [4]:
with open(PROCESSED_DIR / "stage3_audit_summary.json", encoding="utf-8") as f:
    summary = json.load(f)

print("=== BÀI TOÁN 1 (RQ1): THỜI TIẾT KHỚP THỰC & KHUNG GIỜ BOSTON EST ===")
rq1 = summary["business_questions"]["RQ1_weather_and_time"]
print(json.dumps(rq1, indent=2, ensure_ascii=False))

print("\n=== BÀI TOÁN 2 (RQ2): ĐỐI ĐẦU 5 CẶP ĐỐI XỨNG UBER VS LYFT ===")
rq2_df = pd.DataFrame(summary["business_questions"]["RQ2_head_to_head_pairs"])
print(rq2_df.to_string(index=False))

print("\n=== BÀI TOÁN 3 (RQ3): HIỆU NĂNG MÔ HÌNH DỰ BÁO TRÊN TẬP TEST ===")
rq3_df = pd.DataFrame(summary["business_questions"]["RQ3_model_evaluation"]).T
print(rq3_df.to_string())

=== BÀI TOÁN 1 (RQ1): THỜI TIẾT KHỚP THỰC & KHUNG GIỜ BOSTON EST ===
{
  "weather_matched_n": 152043,
  "valid_timestamp_est_n": 178779,
  "rain_count": 24495,
  "dry_count": 127548,
  "rain_mean_price": 16.5859,
  "dry_mean_price": 16.5948,
  "rain_median_price": 13.5,
  "dry_median_price": 13.5,
  "diff_usd": -0.0089,
  "diff_pct": -0.054,
  "welch_t_stat": -0.1365,
  "welch_p_value": 0.8914533072709003,
  "cohens_d": -0.00095,
  "anova_all_f_stat": 0.4668,
  "anova_all_p_value": 0.7054613986324276,
  "anova_all_eta_squared": 8e-06,
  "anova_lyft_f_stat": 0.76,
  "anova_lyft_p_value": 0.516382586197216,
  "anova_lyft_eta_squared": 3.2e-05,
  "time_of_day_means": {
    "Morning": {
      "count": 55039,
      "mean_price": 16.422,
      "mean_surge_all": 1.0127,
      "mean_surge_lyft": 1.0321
    },
    "Afternoon": {
      "count": 36701,
      "mean_price": 16.444,
      "mean_surge_all": 1.0121,
      "mean_surge_lyft": 1.0303
    },
    "Evening": {
      "count": 38704,
      "m